In [1]:
using LatticeDiracOperators
using Gaugefields
using Wilsonloop
using LMP;

[ Info: Precompiling LMP [a75e356b-bbea-40ef-9c9d-31e1efba0018] 
┌ Info: Skipping precompilation due to precompilable error. Importing LMP [a75e356b-bbea-40ef-9c9d-31e1efba0018].
└   exception = Error when precompiling module, potentially caused by a __precompile__(false) declaration in the module.
[ Info: Threads backend loaded with 1 threads


In [2]:
NX = 4
NY = 4
NZ = 4
NT = 4
Nwing = 0
Dim = 4
NC = 3


# Ucpu = Initialize_4DGaugefields(NC,Nwing,NX,NY,NZ,NT,condition = "hot")
# xcpu =Initialize_pseudofermion_fields(Ucpu[1],"Wilson",nowing=true)
# gauss_distribution_fermion!(xcpu)

const pes =  (2, 2, 2, 2)
const cuda = true
const numtrj_in = 2 # the number of operations to take average of performance
const MDsteps_in = 0 # dummy
PEs = pes

U= Initialize_Gaugefields(NC,Nwing,NX,NY,NZ,NT,condition = "hot",cuda=true,blocks=PEs)
# U = Initialize_4DGaugefields(NC,Nwing,NX,NY,NZ,NT,condition = "hot")
x = Initialize_pseudofermion_fields(U[1],"Wilson",nowing=true)
gauss_distribution_fermion!(x)
t = Initialize_pseudofermion_fields(U[1],"Wilson",nowing=true)
gauss_distribution_fermion!(t)
x=x/sqrt(norm2(x))
t=t/sqrt(norm2(t))



params = Dict()
params["Dirac_operator"] = "Wilson"
params["κ"] = 0.14
params["faster version"] = true
params["eps_CG"] = 1.0e-8
params["verbose_level"] = 3

# params1 = Dict()
# params1["Dirac_operator"] = "Wilson"
# params1["κ"] = 0.14
# params1["faster version"] = false
# params1["eps_CG"] = 1.0e-8
# params1["verbose_level"] = 3



Ddag = DdagD_operator(U,x,params)
D= Dirac_operator(U,x,params);
# Dcpu=Dirac_operator(Ucpu,xcpu,params1);

In [3]:
Ritz=Ritz_Layer(D);
# Ritzc=Ritz_Layer(Dcpu);

In [4]:
dl=DenseLinearLayer(2,25,true);

In [5]:
P=Vector{Paths}(undef,9)
for i in 1:9
    if i<=5
        P[i]=Paths([[i-1]])
    else
        P[i]=Paths([[-(i-5)]])
    end
end
dirs = [1, 2, 3, 4, -1, -2, -3, -4]

max_len = 4                  # 만들고 싶은 최대 길이
lengths = Int[]

L = 2                          # 2^0 = 1도 포함하고 싶으면 1부터,
while L <= max_len             # 2부터만이면 L = 2로 시작
    push!(lengths, L)
    L *= 2
end

for d in dirs
    for L in lengths           # L = 1,2,4,8,...(max_len 이하)
        path_vec = fill(d, L)  # [d,d,...,d] (길이 L)
        push!(P, Paths([path_vec]))
    end
end

In [6]:
i=0.1
pl=length(P)
PT=PTLayer(P,U,pl)
PT1=PTLayer(P,U,pl)
PT2=PTLayer(P,U,pl)
PT3=PTLayer(P,U,pl)
PT4=PTLayer(P,U,pl)
PT.W=i*PT.W
PT1.W=i*PT1.W
PT2.W=i*PT2.W
PT3.W=i*PT3.W
PT4.W=i*PT4.W;

In [7]:
Norm=NormLayer()

NormLayer(Identity(), 1.0)

In [8]:
V=deepcopy(x)
gauss_distribution_fermion!(V)

In [9]:
# grad1=deepcopy(PT4.W)
fe=fermionfield_Layer(x,2);
# grad1=deepcopy(fe.x);
grad1=deepcopy(PT3.W)
gram=GramSchmidt_Layer([V]);


In [10]:
layers=[fe,dl,PT3,Norm,Ritz]
f=layers([x])

32.72893746115614

In [11]:
epsilon=1e-8


for i in 1:25
    for u in 1:4
        for v in 1:4
            local layer2=deepcopy(layers)
            layer2[3].W[i][u,v]+=epsilon
            y1=layer2([x])
            grad1[i][u,v]=(y1-f)/epsilon
        end
    end
end ;
# for i in 1:2
#     for k in 1:length(x)

#         local layer2=deepcopy(layers)
#         layer2[1].x[i][k]+=epsilon
#         y1=layer2([x])
#         grad1[i][k]=(y1-f)/epsilon
#     end 

# end ;


In [12]:
γ=Clifford()
 u ,o= forward_propagation(layers, x,t)
delta=deepcopy(u)
backward_propagation!(layers, u,1,delta)

 d = gradient(layers[3], layers[3].activation, u[3], delta[4],γ);

LoadError: UndefRefError: access to undefined reference

In [13]:
typeof(delta[1])

Vector{WilsonFermion_4D_accelerator{3, CuArray{ComplexF64, 4, DeviceMemory}, 4, :cuda, CuArray{ComplexF64, 2, DeviceMemory}, CuArray{ComplexF64, 4, DeviceMemory}}} (alias for Array{LatticeDiracOperators.Dirac_operators.WilsonFermion_4D_accelerator{3, CUDA.CuArray{Complex{Float64}, 4, CUDA.DeviceMemory}, 4, :cuda, CUDA.CuArray{Complex{Float64}, 2, CUDA.DeviceMemory}, CUDA.CuArray{Complex{Float64}, 4, CUDA.DeviceMemory}}, 1})

In [14]:
length(delta)

5

In [15]:
println(grad1[1]-d[1][1])

LoadError: UndefVarError: `d` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [16]:
grad1[3]

4×4 Spin_matrix_GPU:
  0.0292218+0.0im  0.0370235+0.0im  -0.00507114+0.0im    0.0409806+0.0im
 -0.0263441+0.0im  0.0197645+0.0im   -0.0199741+0.0im    -0.011449+0.0im
 -0.0575696+0.0im  0.0497884+0.0im    0.0102574+0.0im   -0.0774811+0.0im
  0.0323787+0.0im  0.0542649+0.0im   0.00761844+0.0im  -0.00320242+0.0im

In [17]:
d[1][3]

LoadError: UndefVarError: `d` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [18]:
opt=Adam()
opt.α=0.1

0.1

In [19]:
@time train(opt,layers,[U],params,epochs=1)

LoadError: UndefRefError: access to undefined reference